In [ ]:
# The goal is to train a classifier that can predict the `Survived` column based on the other columns._
# first try was in ch2/try-it-out/main.ipynb

import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
from sklearn.model_selection import train_test_split

file_path = "./Titanic-Dataset.csv"
titanic_full: pd.DataFrame = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "yasserh/titanic-dataset",
  file_path,
)

train_data, test_data = train_test_split(
    titanic_full, test_size=0.2, stratify=titanic_full["Pclass"], random_state=42
)

# train_data = train_data.set_index("PassengerId")
# test_data = test_data.set_index("PassengerId")
train_data = train_data
test_data = test_data

train_data.info()


<class 'pandas.core.frame.DataFrame'>
Index: 712 entries, 820 to 144
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  712 non-null    int64  
 1   Survived     712 non-null    int64  
 2   Pclass       712 non-null    int64  
 3   Name         712 non-null    object 
 4   Sex          712 non-null    object 
 5   Age          574 non-null    float64
 6   SibSp        712 non-null    int64  
 7   Parch        712 non-null    int64  
 8   Ticket       712 non-null    object 
 9   Fare         712 non-null    float64
 10  Cabin        161 non-null    object 
 11  Embarked     710 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 72.3+ KB


In [45]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
import numpy as np

num_pipeline = Pipeline([
	("imputer", SimpleImputer(strategy="median")),
])

num_freq_and_cat_encode = Pipeline([
	("freq_imputer", SimpleImputer(strategy="most_frequent")),
	("cat_encoder", OneHotEncoder())
])

ord_cat_encode = Pipeline([
	("ord_cat_encoder", OrdinalEncoder())
])

num_scaler = Pipeline([
	# Оригинал условно 0, 7, 14, 512; после log1p: 0, 2.1, 2.7, 6.2; После std: -0.8, -0.2, 0.1, 1.9
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),  
    ("scaler", StandardScaler()), 
])

preprocess_pipeline = ColumnTransformer([
        ("num", num_pipeline, ["Age"]),
		("num_freq", num_freq_and_cat_encode, ["Embarked"]),
		("ord_cat", ord_cat_encode, ["Sex"]),
		("num_scaler", num_scaler, ["Fare"])
    ])

# Преобразовываем данные
X_train = preprocess_pipeline.fit_transform(train_data)
# Получаем лейблы
y_train = train_data["Survived"]

# Отображение
cols = preprocess_pipeline.get_feature_names_out()
pd.DataFrame(X_train, columns=cols).info() # Return to Dataframe

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 712 entries, 0 to 711
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   num__Age              712 non-null    float64
 1   num_freq__Embarked_C  712 non-null    float64
 2   num_freq__Embarked_Q  712 non-null    float64
 3   num_freq__Embarked_S  712 non-null    float64
 4   ord_cat__Sex          712 non-null    float64
 5   num_scaler__Fare      712 non-null    float64
dtypes: float64(6)
memory usage: 33.5 KB


In [55]:
# Впервый раз у меня выходили неплохие значения на LogisticRegression был mean как 0.804871
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
scores = cross_val_score(model, X_train, y_train,
    scoring="accuracy", cv=10)
# На этот раз mean как 0.786541
pd.Series(scores).describe()

count    10.000000
mean      0.786541
std       0.048959
min       0.718310
25%       0.756162
50%       0.788732
75%       0.814065
max       0.873239
dtype: float64

In [ ]:
# Автор книги сразу перешел к RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier

forest_clf = RandomForestClassifier(n_estimators=100, random_state=42)
forest_clf.fit(X_train, y_train)
forest_scores = cross_val_score(forest_clf, X_train, y_train, cv=10)
# В моем случае это 0.7794992175273865
forest_scores.mean()

0.7794992175273865

In [ ]:
# также автор предлагал после этого попробовать SVC
from sklearn.svm import SVC

svm_clf = SVC(gamma="auto")
svm_scores = cross_val_score(svm_clf, X_train, y_train, cv=10)
# У меня вышло как 0.77533255086072
svm_scores.mean()

0.77533255086072